# Historical feature formulas
The original formulas are retained. Training pairs are deduplicated by first occurrence, matching Experiment 01.

In [ ]:
import random
from math import sqrt
from dateutil.parser import parse


### Cache repeated relationship queries


In [ ]:
def memoize(function):
  memo = {}
  def wrapper(*args):

    if args in memo:
      return memo[args]
    else:
      rv = function(*args)
      memo[args] = rv
      return rv
  return wrapper



### Compare event participants


In [ ]:
def get_event_sim_by_users(id1, id2, exclude):
    set1 = attendance.positive_users(id1)
    set2 = attendance.positive_users(id2)

    intersection = set1.intersection(set2)
    s = float(len(intersection))
    if exclude in intersection:
        s -= 1

    s /= min(len(set1), len(set2))

    return s



### Compare historical event clusters


In [ ]:
def get_event_sim_by_cluster(user, event):
    f = []
    for key in ['user_taste', 'friends_taste', 'user_hates', \
            'friends_hate', 'user_invited']:
        if key in user:
            s = 0
            taste = user[key]
            s += taste['cl0'][event['cl0']] * 8
            s += taste['cl1'][event['cl1']] * 20
            s += taste['cl2'][event['cl2']] * 40
            f.append(s)
        else:
            f.append(None)
            #f.append(0 if key in ['user_hates', 'friends_hate'] else None)
    return f



### Retrieve user history


In [ ]:
@memoize
def get_user_attendance(uid):
    return list(attendance.find({'uid': uid}))



### Retrieve event responses


In [ ]:
def get_event_attendance(eid):
    return list(attendance.find({'eid': int(eid)}))



### Aggregate similarities to user history


In [ ]:
@memoize
def get_event_similarity_by_user_big(uid, eid):
    yes_sim = []
    attend_list_u = get_user_attendance(uid)
    for e2 in attend_list_u:
        if ('yes' in e2 or 'maybe' in e2) and e2['eid'] != eid:
            yes_sim.append(get_event_sim_by_users(eid, e2['eid'], uid))
    if yes_sim:
        return sum(yes_sim) / len(yes_sim)
    return None



### Calculate historical location distance


In [ ]:
def get_location_distance(l1, l2):
    if not l1 or not l2:
        return None

    if isinstance(l1[0], float):
        l1 = [l1]
    if isinstance(l2[0], float):
        l2 = [l2]
    minimum = 100000000
    for a in l1:
        for b in l2:
            d = sqrt((a[0] - b[0]) ** 2 + (a[1] - b[1]) ** 2)
            if d < minimum:
                minimum = d

    return minimum



### Compare content distributions


In [ ]:
def get_event_distance(w1, w2):
    # this uses the cosine similarity
    if not w1 or not w2:
        return None
    w1 = np.array(w1)
    w2 = np.array(w2)
    w1 = np.log(w1 + 1)
    w2 = np.log(w2 + 1)
    s = np.sum(w1 * w2) / (np.sqrt(np.sum(w1 * w1)) * np.sqrt(np.sum(w2 * w2)))

    return s



### Match location names


In [ ]:
def compare_location_string(s1, s2):
    if not s1 or not s2:
        return 0
    s1 = s1.lower()
    s2 = s2.lower()
    jog = ['yogyakarta', 'jogjakarta']
    if s1 in jog and s2 in jog:
        return 1
    return int(s1 == s2)



### Combine geographic matches


In [ ]:
def process_locations(elocs, ulocs):
    same_country = 0
    same_state = 0
    same_city = 0
    for el in elocs:
        for ul in ulocs:
            same_country = max(same_country, \
                compare_location_string(el.get('country'), ul.get('country')))
            same_state = max(same_state, \
                compare_location_string(el.get('state'), ul.get('state')))
            same_city = max(same_city, \
                compare_location_string(el.get('city'), ul.get('city')))
            if compare_location_string(el.get('country'), ul.get('country')) \
                    and same_city == 0:
                if 'city' in el and 'city' in ul:
                    same_city = 0.5
                elif 'city' in el:
                    same_city = 0.25
    return [same_country + same_state + same_city * 2.0]



### Assemble historical candidate features


In [ ]:
def process_events(uid, e_dict):
    # gets a dict of events to determine a user's interest (e_dict) and a uid
    # get the events related to the user from the db
    # e_dict has value for invited
    attend_list_u = get_user_attendance(uid)
    attend_list_ids = [f['eid'] for f in attend_list_u]
    e_list = list(event_info.find({'id':{'$in': list(e_dict.keys())}}))
    user = user_info.find_one({'id': uid})
    attend_dict = {e['eid']: e for e in attend_list_u}

    friends = friends_db.find_one({'uid': int(uid)})
    friend_ids = set(friends['friends'])

    features_dict = {}
    for e in e_list:
        attend_list_e = get_event_attendance(e['id'])
        features = [0, 0, 0, 0]
        for att in attend_list_e:
            if 'yes' in att:
                features[0] += 1
            if 'no' in att:
                features[1] += 1
            if 'maybe' in att:
                features[2] += 1
            if 'invited' in att:
                features[3] += 1
        features.extend([
            features[1] * 1.0 / (features[0] + 1),
            features[2] * 1.0 / (features[0] + 1),
            features[3] * 1.0 / (features[0] + 1),
        ])

        features2 = [0, 0, 0, 0]
        for att in attend_list_e:
            if att['uid'] not in friend_ids:
                continue
            if 'yes' in att:
                features2[0] += 1
            if 'no' in att:
                features2[1] += 1
            if 'maybe' in att:
                features2[2] += 1
            if 'invited' in att:
                features2[3] += 1
        features2.extend([
            features2[1] * 1.0 / (features2[0] + 1),
            features2[2] * 1.0 / (features2[0] + 1),
            features2[3] * 1.0 / (features2[0] + 1),
        ])
        features2.extend([
            features2[0] / (len(friend_ids) + 1.0),
            features2[1] / (len(friend_ids) + 1.0),
            features2[2] / (len(friend_ids) + 1.0),
            features2[3] / (len(friend_ids) + 1.0),
        ])
        features.extend(features2)

        # add distance to event

        features.extend(
            process_locations(
                e.get('newloc2', []),
                user.get('newloc2', [])
            )
        )

        # add age profile
        if user['birth'] != 'None' and isinstance(user['birth'], str) \
                and 'ages' in e:
            d = abs(2013 - int(user['birth']) - e['ages']['mean'])
            features.append(d + int(random.random() * 6))
        else:
            features.append(None)

        # add gender profile
        if user['gender'] != 'None' and isinstance(user['gender'], str) \
                and 'genders' in e:
            features.append((e['genders'][user['gender']] + 1.0) / \
                    (e['genders']['male'] + e['genders']['female'] + 2.0))
        else:
            features.append(None)

        # add event similarity by user
        features.append(get_event_similarity_by_user_big(uid, e['id']))

        # add user to event cluster sim
        features.extend(get_event_sim_by_cluster(user, e))
        # add time to event
        features.append(e['start'] - e_dict[e['id']][1])

        # see if event creator is a friend - ar putea cauza scadere de 0.2%
        features.append(e['creator'] in friend_ids)

        # compare to what the user's friends like
        if 'prototype' in user:
            features.append(get_event_distance(user['prototype'], e['words']))
        else:
            features.append(None)

        if 'prototype' in user and 'prototype_invite' in user:
            v1 = get_event_distance(user['prototype'], e['words'])
            v2 = get_event_distance(user['prototype_invite'], e['words'])
            if v1 and v2:
                features.append(v1 - v2)
            else:
                features.append(None)
        else:
            features.append(None)
        if 'prototype_hate' in user and 'prototype_invite' in user:
            v1 = get_event_distance(user['prototype_hate'], e['words'])
            v2 = get_event_distance(user['prototype_invite'], e['words'])
            if v1 and v2:
                features.append(v1 - v2)
            else:
                features.append(None)
        else:
            features.append(None)
        if 'prototype_hate' in user and 'prototype' in user:
            v1 = get_event_distance(user['prototype_hate'], e['words'])
            v2 = get_event_distance(user['prototype'], e['words'])
            if v1 and v2:
                features.append(v1 - v2)
            else:
                features.append(None)
        else:
            features.append(None)

        # add invited flag
        features.append(e_dict[e['id']][0])

        # old locations
        features.append(get_location_distance(e['location'], user['location']))

        features_dict[e['id']] = features

    # normalize location score
    return features_dict



### Construct five fixed feature partitions


In [ ]:
def get_crossval_data():
    cache_path = OUT / 'five_fold_features_utc.pkl'
    if cache_path.exists():
        return pd.read_pickle(cache_path)
    train = pd.read_csv(ROOT / 'data/raw/train.csv')
    folds = pd.read_csv(ROOT / 'data/folds.csv')
    fold_map = folds.groupby('user').fold.first().to_dict()
    train_dict = {}
    duplicates = set([])
    for record in train.iterrows():
        record = record[1]
        uid = record['user']

        # check for duplicates
        key = (uid, record['event'])
        if key in duplicates:
            continue
        duplicates.add(key)

        if uid not in train_dict:
            train_dict[uid] = []
        train_dict[uid].append({
            'eid': record['event'],
            'invited': record['invited'],
            'interested': record['interested'],
            'not_interested': record['not_interested'],
            'timestamp': parse(record['timestamp']).timestamp(),  # Use UTC seconds consistently for exposure and event timestamps.
        })

    splits = []
    for i in range(1, 6):
        X = []
        Y1 = []
        Y2 = []
        results = {}
        keys = []
        count = train['user'].count()
        for uid, events in train_dict.items():
            if fold_map[uid] != i:
                continue
            e_dict = {e['eid']: (e['invited'], e['timestamp']) for e in events}
            features_dict = process_events(uid, e_dict)
            if random.random() < 0.1:
                print(len(X) * 100.0 / count)
            results[uid] = []
            for e in events:
                eid = e['eid']
                X.append(features_dict[eid])
                Y1.append(e['interested'])
                Y2.append(e['not_interested'])
                keys.append((uid, e['eid']))
                if e['interested']:
                    results[uid].append(e['eid'])

        splits.append((X, Y1, Y2, results, keys))

    pd.to_pickle(splits, cache_path)
    return splits
